In [1]:
# https://github.com/Stable-Baselines-Team/rl-colab-notebooks/blob/sb3/dqn_sb3.ipynb

import gymnasium as gym
import numpy as np
import torch as th
import matplotlib.pyplot as plt

from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

In [4]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

In [5]:
tensorboard_log = "data/tb/"

In [6]:

### Aqui es donde creamos el modelo
dqn_model = DQN(
    "MlpPolicy", #politica perceptrón multicapa. tambien puede ser CNNPolicy.
    env,
    verbose=1,
    train_freq=16,
    gradient_steps=8,
    gamma=0.99,
    exploration_fraction=0.2,
    exploration_final_eps=0.07,
    target_update_interval=600,
    learning_starts=1000,
    buffer_size=10000,
    batch_size=128,
    learning_rate=4e-3,
    policy_kwargs=dict(net_arch=[256, 256]),
    tensorboard_log=tensorboard_log,
    seed=2,
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
mean_reward, std_reward = evaluate_policy(
    dqn_model,
    dqn_model.get_env(),
    deterministic=True,
    n_eval_episodes=20,
)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward:9.35 +/- 0.65


In [8]:
# Optional: Monitor training in tensorboard
# %load_ext tensorboard
# %tensorboard --logdir $tensorboard_log



In [11]:
!pip install tensorboard


In [8]:
dqn_model.learn(int(1.2e5), log_interval=10)

Logging to data/tb/DQN_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 23.3     |
|    ep_rew_mean      | 23.3     |
|    exploration_rate | 0.991    |
| time/               |          |
|    episodes         | 10       |
|    fps              | 17504    |
|    time_elapsed     | 0        |
|    total_timesteps  | 233      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 26.6     |
|    ep_rew_mean      | 26.6     |
|    exploration_rate | 0.979    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 20791    |
|    time_elapsed     | 0        |
|    total_timesteps  | 533      |
----------------------------------


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 24.1     |
|    ep_rew_mean      | 24.1     |
|    exploration_rate | 0.972    |
| time/               |          |
|    episodes         | 30       |
|    fps              | 18547    |
|    time_elapsed     | 0        |
|    total_timesteps  | 722      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 22.4     |
|    ep_rew_mean      | 22.4     |
|    exploration_rate | 0.965    |
| time/               |          |
|    episodes         | 40       |
|    fps              | 18286    |
|    time_elapsed     | 0        |
|    total_timesteps  | 898      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21.9     |
|    ep_rew_mean      | 21.9     |
|    exploration_rate | 0.957    |
| time/               |          |
|    episodes       

In [9]:
mean_reward, std_reward = evaluate_policy(dqn_model, dqn_model.get_env(), deterministic=True, n_eval_episodes=20)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward:65.05 +/- 51.38


In [10]:
# Set up fake display; otherwise rendering will fail
import os
os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'

sh: Xvfb: command not found


In [11]:
import base64
from pathlib import Path

from IPython import display as ipythondisplay


def show_videos(video_path="", prefix=""):
    """
    Taken from https://github.com/eleurent/highway-env

    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append(
            """<video alt="{}" autoplay 
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [12]:
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv


def record_video(
    env_id,
    model,
    video_length=500,
    prefix="",
    video_folder="videos/",
):
    """
    :param env_id: (str)
    :param model: (RL model)
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    # Start the video at step=0 and record 500 steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset()
    for _ in range(video_length):
        action, _ = model.predict(obs, deterministic=False)
        obs, _, _, _ = eval_env.step(action)

    # Close the video recorder
    eval_env.close()

In [14]:
!pip install 'gymnasium[other]'


  Using cached imageio-2.37.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached python_dotenv-1.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 21.6 MB/s eta 0:00:00a 0:00:01
Using cached imageio-2.37.0-py3-none-any.whl (315 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 20.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 18.7 MB/s eta 0:00:00
Using cached python_dotenv-1.1.0-py3-none-any.whl (20 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.1.0
    Uninstalling pillow-11.1.0:
      Successfully uninstalled pillow-11.1.0


In [15]:
record_video("CartPole-v1", dqn_model, video_length=500, prefix="dqn-cartpole")

Saving video to /Users/adriandelpozo/Desktop/MSc_BIG_DATA/MACHINE_LEARNING_II/reinforcement_learning/Codigos RL Moodle ML II/8_DQN/videos/dqn-cartpole-step-0-to-step-500.mp4
MoviePy - Building video /Users/adriandelpozo/Desktop/MSc_BIG_DATA/MACHINE_LEARNING_II/reinforcement_learning/Codigos RL Moodle ML II/8_DQN/videos/dqn-cartpole-step-0-to-step-500.mp4.
MoviePy - Writing video /Users/adriandelpozo/Desktop/MSc_BIG_DATA/MACHINE_LEARNING_II/reinforcement_learning/Codigos RL Moodle ML II/8_DQN/videos/dqn-cartpole-step-0-to-step-500.mp4



MoviePy - Done !
MoviePy - video ready /Users/adriandelpozo/Desktop/MSc_BIG_DATA/MACHINE_LEARNING_II/reinforcement_learning/Codigos RL Moodle ML II/8_DQN/videos/dqn-cartpole-step-0-to-step-500.mp4


In [16]:
show_videos("videos", prefix="dqn")